# LUNA16 Pipeline

Alternative to the LIDC-IDRI DICOM pipeline. Uses LUNA16 MHD/RAW CT scans
with malignancy labels fetched from LIDC-IDRI XML annotations via pylidc.

**Prerequisites:** Run cells in order. GPU not required for this notebook.

| Variable | Default | Description |
|---|---|---|
| `SUBSET_IDS` | `[0,1,2,3]` | Which subsets to use (0-9, each ~6-7 GB) |
| `PATCH_SIZE` | `64` | 3D patch size in voxels |
| `TOL_MM` | `3.0` | Radius for LIDC annotation matching |


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os
REPO_PATH = '/content/lung-nodule-fusion-xai'
if not os.path.exists(REPO_PATH):
    subprocess.run([
        'git', 'clone',
        'https://github.com/YOUR_USERNAME/lung-nodule-fusion-xai.git',
        REPO_PATH
    ], check=True)
os.chdir(REPO_PATH)
import sys; sys.path.insert(0, REPO_PATH)
print('Repo ready:', REPO_PATH)


In [ ]:
!pip install -q pylidc==0.2.2 SimpleITK==2.3.1 scikit-learn==1.5.0 \
               pandas==2.2.2 pyarrow==16.1.0 nibabel==5.2.1


In [ ]:
# === USER CONFIG ===
DRIVE_BASE  = '/content/drive/MyDrive/Kanker Kanker apa yg Kanker'
SUBSET_DIR  = '/content/luna16'          # local extraction target (Colab disk)
SUBSET_IDS  = [0, 1, 2, 3]              # subsets to use; add more if disk allows
PATCH_SIZE  = 64                         # voxels per side
TOL_MM      = 3.0                        # LIDC annotation match tolerance

DRIVE_PART1     = f'{DRIVE_BASE}/luna16 part 1'
DRIVE_PART2     = f'{DRIVE_BASE}/luna16 part 2'
CANDIDATES_CSV  = f'{DRIVE_PART1}/candidates.csv'
METADATA_CSV    = f'{DRIVE_BASE}/metadata/metadata.csv'
LIDC_PATH       = f'{DRIVE_BASE}/lidc_idri'
INTERIM_DIR     = '/content/drive/MyDrive/lung_nodule_interim_luna16'
PROCESSED_DIR   = '/content/drive/MyDrive/lung_nodule_processed_luna16'
print('Config OK')


In [ ]:
# Configure pylidc (needed for malignancy label lookup)
import configparser, os
cfg = configparser.ConfigParser()
cfg['pylidc'] = {'dicom_path': LIDC_PATH}
with open(os.path.expanduser('~/.pylidcrc'), 'w') as f:
    cfg.write(f)
import pylidc as pl
n_scans = pl.query(pl.Scan).count()
print(f'pylidc configured: {n_scans} scans found')


In [ ]:
# Step 1: Extract LUNA16 subset zips from Drive
# Each subset ~6-7 GB; check Colab disk first: df -h /content
!df -h /content

from src.data_loading.luna16_loader import extract_subsets
extract_subsets(
    output_dir=SUBSET_DIR,
    subset_ids=SUBSET_IDS,
    drive_part1=DRIVE_PART1,
    drive_part2=DRIVE_PART2,
)
print('Subsets ready')
!ls {SUBSET_DIR}


In [ ]:
# Step 2 (optional): Extract lung segmentation masks
# Useful for Route B auto-segmentation; skip if not needed
from src.data_loading.luna16_loader import extract_lung_masks
SEG_DIR = '/content/luna16_seg'
extract_lung_masks(output_dir=SEG_DIR, drive_part1=DRIVE_PART1)
print('Lung masks ready at', SEG_DIR)


In [ ]:
# Step 3: Build labels DataFrame
# This links LUNA16 candidate positions → LIDC malignancy labels via pylidc
# Runtime: ~1-3 h depending on subset count
import logging
logging.basicConfig(level=logging.INFO)

from src.data_loading.luna16_loader import load_and_split_luna16

df = load_and_split_luna16(
    subset_dir=SUBSET_DIR,
    candidates_csv=CANDIDATES_CSV,
    metadata_csv=METADATA_CSV,
    lidc_path=LIDC_PATH,
    output_dir=INTERIM_DIR,
    processed_path=PROCESSED_DIR,
    n_folds=5,
    seed=42,
)
print(f'Total: {len(df)} | Malignant: {(df.label==1).sum()} | Benign: {(df.label==0).sum()}')
df.head()


In [ ]:
# Step 4: Verify patch shapes
import numpy as np

sample = df.iloc[0]
patch = np.load(sample['patch_path'])
mask  = np.load(sample['mask_path'])
print('Patch shape:', patch.shape, '| dtype:', patch.dtype)
print('Mask shape: ', mask.shape,  '| unique:', np.unique(mask))
print('Label:', sample['label'], '| diameter_mm:', sample.get('diameter_mm', 'N/A'))


In [ ]:
# Step 5: Visualise a few nodules
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(df): break
    row = df.iloc[i]
    patch = np.load(row['patch_path'])
    mid = patch.shape[0] // 2
    ax.imshow(patch[mid], cmap='gray')
    ax.set_title(f"{'Mal' if row['label']==1 else 'Ben'} | {row.get('diameter_mm', 0):.1f}mm")
    ax.axis('off')
plt.suptitle('LUNA16 Nodule Patches (centre axial slice)')
plt.tight_layout()
plt.savefig(f'{PROCESSED_DIR}/sample_patches.png', dpi=150)
plt.show()


In [ ]:
# Step 6: Fold distribution
import matplotlib.pyplot as plt

fold_counts = df.groupby(['fold', 'label']).size().unstack(fill_value=0)
fold_counts.columns = ['Benign', 'Malignant']
fold_counts.plot(kind='bar', figsize=(8, 4), title='LUNA16 — Fold Distribution')
plt.xlabel('Fold'); plt.ylabel('Count')
plt.tight_layout()
plt.savefig(f'{PROCESSED_DIR}/fold_distribution.png', dpi=150)
plt.show()
print('labels CSV saved to:', PROCESSED_DIR)


## Next Steps

After this notebook completes, use the generated labels CSV with the rest of the pipeline:

- **Radiomics**: `radiomics_extraction.ipynb` — point `labels_df` to `labels_luna16.csv`
- **CNN Training**: `phase1_cnn_benchmark.ipynb` — set `PROCESSED_DIR` to `lung_nodule_processed_luna16`
- **Fusion**: `phase2_fusion.ipynb` — same

The patch schema is identical to the LIDC-IDRI pipeline so all downstream code works unchanged.
